# Artwork Multi-Method Eval — EA, KVzip, Finch (CPT vs Full)
Results + checkpoint saved to **Google Drive**.

**Code parity version: Matches exactly the structure and style of the previous working checkpointed notebook.**


## Step 0.5 — GitHub PAT

In [ ]:
import getpass, os
manual = getpass.getpass("GitHub Classic PAT (leave empty to use Colab Secret): ").strip()
if manual:
    gh_token = manual
else:
    try:
        from google.colab import userdata
        gh_token = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        gh_token = os.environ.get("GITHUB_TOKEN", "").strip()
print("Token loaded:", bool(gh_token))

## Step 1 — Clone repos

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

BENCH_URL = "https://github.com/physical168/kv-compression-benchmark.git"
CE_URL    = "https://github.com/GabrieleSanmartino/CompressionExperiments.git"
BENCH_DIR = Path("/content/kv-compression-benchmark")
CE_DIR    = Path("/content/CompressionExperiments")

def clone(url, out, token, name):
    r = subprocess.run(["git","clone","--depth","1",url,str(out)], capture_output=True, text=True)
    if r.returncode == 0: print(name, "public clone OK"); return True
    print(name, "public clone failed"); print((r.stderr or "")[:300])
    if not token: return False
    r2 = subprocess.run(["git","clone","--depth","1",url.replace("https://github.com/",f"https://{token}@github.com/"),str(out)], capture_output=True, text=True)
    if r2.returncode == 0: print(name, "token clone OK"); return True
    print(name, "token clone failed"); print((r2.stderr or "")[:300]); return False

_tok = (globals().get("gh_token") or "").strip() or None
os.chdir("/content")
for p in [BENCH_DIR, CE_DIR]:
    if p.exists(): shutil.rmtree(p)
clone(BENCH_URL, BENCH_DIR, _tok, "BENCH")
clone(CE_URL, CE_DIR, _tok, "CE")


## Step 1.2 — Install dependencies & LFS

In [ ]:
import subprocess, sys, os
from pathlib import Path
_ce = Path("/content/CompressionExperiments")

# Git LFS
subprocess.run(["apt-get", "install", "git-lfs"], capture_output=True)
if _ce.is_dir():
    subprocess.run(["git", "lfs", "install"], cwd=str(_ce))
    subprocess.run(["git", "lfs", "pull"], cwd=str(_ce))

subprocess.check_call([sys.executable,"-m","pip","install","-q","-U","pip"])
subprocess.check_call([sys.executable,"-m","pip","install","-q",
    "transformers>=5.0","accelerate","bitsandbytes","pandas","pyyaml","scikit-learn","matplotlib","tqdm","pillow"])

# Install kvpress
_kvp = _ce / "kvpress"
if _kvp.is_dir():
    subprocess.check_call([sys.executable,"-m","pip","install",str(_kvp)])
    if str(_kvp) not in sys.path: sys.path.insert(0, str(_kvp))
else:
    subprocess.check_call([sys.executable,"-m","pip","install","kvpress"])

import kvpress
print("kvpress version:", getattr(kvpress, "__version__", "unknown"))


## Step 1.5 — Mount Google Drive

In [ ]:
from google.colab import drive
import os, pandas as pd
from pathlib import Path
drive.mount("/content/drive", force_remount=False)
DRIVE_OUT        = "/content/drive/MyDrive/kv-compression-benchmark/ce_artwork_all_results"
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH  = Path(DRIVE_OUT) / "checkpoint_all.csv"
RESULTS_ROOT     = Path(DRIVE_OUT) / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)


## Step 2 — Prepare dataset

In [ ]:
import os, shutil
from pathlib import Path
CE_ART = Path("/content/CompressionExperiments/experiment_manager/datasets/artwork")
IMAGES_DIR = CE_ART / "images"
DATASET_PATH = CE_ART / "paintings.csv"

if not DATASET_PATH.is_file():
    shutil.copy2("/content/kv-compression-benchmark/paintings.csv", DATASET_PATH)

print(f"Images ready at: {IMAGES_DIR} | count: {len(list(IMAGES_DIR.glob('*')))}")


## Step 3 — Load model + kvpress patches

In [ ]:
import sys, importlib.util, torch, transformers
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "llava-hf/llama3-llava-next-8b-hf"
MODEL_TAG = MODEL_ID.split("/")[-1]
bnb_cfg = BitsAndBytesConfig(load_in_8bit=True, llm_int8_enable_fp32_cpu_offload=True)

print(f"Loading {MODEL_ID} ...")
processor = LlavaNextProcessor.from_pretrained(MODEL_ID)
model = LlavaNextForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", quantization_config=bnb_cfg)

_patch = "/content/kv-compression-benchmark/benchmarks/artwork_eval/llava_kvpress_patch.py"
_spec = importlib.util.spec_from_file_location("patch", _patch)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
_mod.apply_kvpress_compatibility_patches(model)
print("Model ready.")


## Step 4 — Configuration

In [ ]:
from kvpress import ExpectedAttentionPress, KVzipPress, FinchPress

MAX_ROWS = 10
MAX_NEW_TOKENS = 50
COMPRESSION_RATIOS = [0.4, 0.8, 0.95]

# method_key -> (PressClass, use_cpt)
EXPERIMENT_CONFIGS = {
    # "EA": (ExpectedAttentionPress, True), # Excluded: merge later
    "KVzip": (KVzipPress, True),
    "Finch_Full": (FinchPress, False),
    "Finch_CPT": (FinchPress, True),
}

PRESS_REGISTRY = {
    "ExpectedAttentionPress": ExpectedAttentionPress,
    "KVzipPress": KVzipPress,
    "FinchPress": FinchPress,
}

_EXTRACT_SUFFIX = (" (be concise, no explanation, no introductory text, just the answer,"
                   " output datatype: STRING, do not repeat the datatype in the answer) ?")

def build_answer_prefix(row, question: str, is_boolean: bool, use_cpt: bool) -> str:
    context = "" if use_cpt else f"This painting is titled '{row['title']}', created around {row['inception']}. It belongs to the {row['movement']} movement and {row['genre']} genre. "
    if is_boolean:
        return (f"{context}Answer the following question based on the image with '1' or '0'."
                f" Do not add any other comments. {question}\nAnswer: ")
    return (f"{context}Answer the following question based on the image."
            f" Do not add any other comments. {question + _EXTRACT_SUFFIX}\nAnswer: ")


## Step 5 — Load dataset & queries

In [ ]:
import yaml, pandas as pd
from urllib.parse import unquote
with open("/content/kv-compression-benchmark/benchmarks/artwork_eval/configs/image_queries.yaml") as f:
    q_cfg = yaml.safe_load(f)["artwork"]
queries_plan = [(q, True) for q in q_cfg["filter_queries"]] + [(q, False) for q in q_cfg["extract_queries"]]

df = pd.read_csv(DATASET_PATH).head(MAX_ROWS)
df["record_id"] = range(len(df))
def resolve_image(url: str) -> str:
    tail = url.split("/")[-1].split("?")[0]
    return str(IMAGES_DIR / unquote(tail))
df["image_path"] = df["image_url"].apply(resolve_image)


## Step 6 — Multi-Method Checkpointed Inference

In [ ]:
import csv as _csv, gc, os, time, torch
from PIL import Image

_tok = getattr(processor, "tokenizer", processor)

def run_generate_vision(image_path, answer_prefix, PressCls, ratio):
    press = PressCls(compression_ratio=float(ratio))
    if hasattr(press, "update_model_and_tokenizer"):
        press.update_model_and_tokenizer(model, _tok)
    image = Image.open(image_path).convert("RGB")
    conv  = [{"role":"user","content":[{"type":"image"},{"type":"text","text":answer_prefix}]}]
    prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    gen_kw = {"max_new_tokens": MAX_NEW_TOKENS, "pad_token_id": _tok.pad_token_id}
    with torch.no_grad(), press(model):
        out = model.generate(**inputs, **gen_kw)
    return _tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def load_done(path):
    if not os.path.exists(path): return set()
    ck = pd.read_csv(path)
    done = set()
    for _, r in ck.iterrows():
        if str(r.get("error","")).strip().lower() in ("","nan"):
            done.add((str(r["method"]), str(float(r["ratio"])), int(r["record_id"]), str(r["query"])))
    return done

def append_ck(path, row):
    fields = ["method","ratio","record_id","query","answer","error"]
    newfile = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = _csv.DictWriter(f, fieldnames=fields)
        if newfile: w.writeheader()
        w.writerow(row)

done_keys = load_done(CHECKPOINT_PATH)
print(f"Resuming: {len(done_keys)} already done.")

for m_name, (Cls, use_cpt) in EXPERIMENT_CONFIGS.items():
    print(f"--- Processing {m_name} ---")
    for ratio in COMPRESSION_RATIOS:
        for _, row in df.iterrows():
            for qtext, is_bool in queries_plan:
                key = (m_name, str(float(ratio)), int(row["record_id"]), qtext)
                if key in done_keys: continue
                
                ap = build_answer_prefix(row, qtext, is_bool, use_cpt)
                err = ans = ""
                try:
                    ans = run_generate_vision(row["image_path"], ap, Cls, ratio)
                except Exception as e:
                    err = str(e)[:200]
                    print(f"[err] rid={row.record_id} {m_name} r={ratio}: {err}")
                
                append_ck(CHECKPOINT_PATH, {"method":m_name,"ratio":ratio,"record_id":row.record_id,"query":qtext,"answer":ans,"error":err})
                if not err: done_keys.add(key)
                gc.collect(); torch.cuda.empty_cache()
print("All tasks finished.")


## Step 7 — Merge EA results & Reconstruct Tree

In [ ]:
import pandas as pd
from pathlib import Path

ck_all = pd.read_csv(CHECKPOINT_PATH)
OLD_DRIVE_OUT = "/content/drive/MyDrive/kv-compression-benchmark/ce_artwork_results"
OLD_EA_PATH = Path(OLD_DRIVE_OUT) / "checkpoint.csv"

if OLD_EA_PATH.is_file():
    print("Merging old EA results...")
    ea_df = pd.read_csv(OLD_EA_PATH).rename(columns={'press': 'method'})
    ea_df['method'] = 'EA'
    ck_combined = pd.concat([ea_df, ck_all], ignore_index=True).drop_duplicates(subset=['method','ratio','record_id','query'], keep='first')
else:
    print("No old EA checkpoint found.")
    ck_combined = ck_all

ck_ok = ck_combined[ck_combined.error.fillna("").astype(str).str.strip().isin(["","nan"])].copy()

for (m, r), grp in ck_ok.groupby(["method", "ratio"]):
    ratio_tag = f"{float(r):.2f}"
    out_dir = RESULTS_ROOT / "artwork" / f"Llava3-8b-{m}" / ratio_tag
    out_dir.mkdir(parents=True, exist_ok=True)
    grp[["record_id","query","answer"]].to_csv(out_dir / "results.csv", index=False)
    print(f"Tree ready for {m} R{ratio_tag}")


## Step 8 — Evaluate

In [ ]:
import sys
sys.path.append("/content/kv-compression-benchmark/benchmarks/artwork_eval")
from evaluation.evaluator import EvaluationManager
mgr = EvaluationManager(config_path="/content/kv-compression-benchmark/benchmarks/artwork_eval/evaluation/evaluation_config.yaml", results_dir=str(RESULTS_ROOT))
res = mgr.evaluate_all()
display(res.groupby(["model_tag","ratio"])[["precision","recall","f1"]].mean())
